# 🌿 Plant Disease Detection using CNN
## Notebook 4 — Improving Real-World Accuracy (Field Fine-Tuning)

### Why this notebook?
Our MobileNetV2 scores **96.8% on lab photos** but only **~20% on real field / internet photos** (Notebook 3). It learned the *lab photo style* (one leaf, grey background) instead of only the disease. This is called **domain shift**.

### What we do
1. Take the trained MobileNetV2 from Notebook 2.
2. **Fine-tune** it on a **mix of photos**:
   - **PlantDoc *train* folder**: ~2,300 real field/internet photos (new for the model)
   - **150 PlantVillage lab photos per class**, so the model doesn't forget what it already knows
3. Evaluate **before vs after** on:
   - **PlantDoc *test* folder**: real photos kept separate and **never used for training** (fair field score)
   - **PlantVillage test set**: to check lab accuracy stays high

The improved model is saved as `mobilenetv2_field.keras`, and the web app (Notebook 3) uses it automatically.

**Run Notebook 2 first.** Then `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (~25–35 minutes).

In [ ]:
# ============================================================
# SETUP: imports, settings and project folders
# ============================================================
import os, sys, json, time, random, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42                     # fixed "random seed" -> same results every run
random.seed(SEED); np.random.seed(SEED)

IMG_SIZE = 224                # every image is resized to 224 x 224 pixels
BATCH_SIZE = 32               # how many images the model sees at once

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Google Drive keeps our trained models & results safe even if Colab disconnects
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        PROJECT_DIR = '/content/drive/MyDrive/plant-disease-project'
    except Exception as e:
        # Drive permission failed (tip: tick "Select all" on the Google permission screen).
        # We continue anyway, but results are only kept until this Colab session ends.
        print('WARNING: Google Drive could not be connected:', e)
        print('Saving to /content/outputs instead - download the results before closing Colab!')
        PROJECT_DIR = '/content/outputs'
    DATA_ROOT = '/content/data'              # fast local disk for the images
else:
    PROJECT_DIR = os.path.abspath('../outputs')
    DATA_ROOT = os.path.abspath('../data')

MODELS_DIR  = os.path.join(PROJECT_DIR, 'models')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FIG_DIR     = os.path.join(RESULTS_DIR, 'figures')
for d in [MODELS_DIR, RESULTS_DIR, FIG_DIR, DATA_ROOT]:
    os.makedirs(d, exist_ok=True)

DATASET_DIR = os.path.join(DATA_ROOT, 'PlantVillage-Dataset', 'raw', 'color')
print('Running in Colab:', IN_COLAB)
print('Results will be saved in:', PROJECT_DIR)

In [ ]:
RETRAIN = False            # True = fine-tune again even if mobilenetv2_field.keras exists
EPOCHS_FIELD = 12          # max fine-tuning rounds (early stopping usually ends sooner)
LAB_PER_CLASS = 150        # lab photos per class mixed into training
FIELD_REPEAT = 3           # field photos are repeated 3x per epoch (each time differently augmented)
UNFREEZE_LAYERS = 60       # how many top MobileNetV2 layers are allowed to change

In [ ]:
# ============================================================
# DOWNLOAD THE PLANTVILLAGE DATASET (about 1 GB, takes ~2-5 minutes)
# Source: the official GitHub copy by S. P. Mohanty (author of our main reference paper)
# We download only the 'raw/color' folder = 54,305 original colour leaf photos.
# ============================================================
REPO_URL = 'https://github.com/spMohanty/PlantVillage-Dataset'
repo_dir = os.path.join(DATA_ROOT, 'PlantVillage-Dataset')

if os.path.isdir(DATASET_DIR) and len(os.listdir(DATASET_DIR)) == 38:
    print('Dataset already downloaded - skipping.')
else:
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    REPO_URL, repo_dir], check=True)
    subprocess.run(['git', '-C', repo_dir, 'sparse-checkout', 'set', 'raw/color'], check=True)
    print(f'Download finished in {time.time() - t0:.0f} seconds')

n_images = sum(len(files) for _, _, files in os.walk(DATASET_DIR))
print('Class folders:', len(os.listdir(DATASET_DIR)))
print('Total images :', n_images)     # should print 54305

In [ ]:
# ============================================================
# HELPER: turn folder names into readable names
# 'Tomato___Early_blight'  ->  'Tomato - Early blight'
# ============================================================
def pretty_name(folder_name):
    crop, disease = folder_name.split('___')
    crop = crop.replace('_', ' ').replace(',', '').strip()
    disease = disease.replace('_', ' ').strip()
    disease = disease[0].upper() + disease[1:]
    return f'{crop} - {disease}'

def crop_of(folder_name):
    return folder_name.split('___')[0].replace('_', ' ').replace(',', '').strip()

In [ ]:
# ============================================================
# BUILD A TABLE OF ALL IMAGES: one row per image -> (file path, class)
# ============================================================
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
CLASS_NAMES = sorted(d.name for d in Path(DATASET_DIR).iterdir() if d.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}

records = []
for c in CLASS_NAMES:
    for p in sorted((Path(DATASET_DIR) / c).iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            # store the path RELATIVE to the dataset folder (works in any session)
            records.append((f'{c}/{p.name}', c, CLASS_TO_ID[c]))

df = pd.DataFrame(records, columns=['filepath', 'label', 'label_id'])
print(f'{len(df)} images in {NUM_CLASSES} classes')

# Save the list of class names - the app needs it later to turn numbers back into names
with open(os.path.join(MODELS_DIR, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f, indent=1)
df.head()

In [ ]:
SHORT = [pretty_name(c) for c in CLASS_NAMES]

In [ ]:
# ============================================================
# SPLIT INTO TRAIN (70%) / VALIDATION (15%) / TEST (15%)
# 'stratify' keeps the same class proportions in every part,
# so rare classes (e.g. Potato healthy, 152 images) appear in all three.
# The split is saved, so every notebook uses EXACTLY the same images.
# ============================================================
from sklearn.model_selection import train_test_split

split_files = {s: os.path.join(RESULTS_DIR, f'split_{s}.csv') for s in ['train', 'val', 'test']}

if all(os.path.exists(f) for f in split_files.values()):
    train_df, val_df, test_df = (pd.read_csv(split_files[s]) for s in ['train', 'val', 'test'])
    print('Loaded the saved split.')
else:
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label_id'],
                                         random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label_id'],
                                       random_state=SEED)
    for s, part in zip(['train', 'val', 'test'], [train_df, val_df, test_df]):
        part.reset_index(drop=True).to_csv(split_files[s], index=False)
    train_df, val_df, test_df = (pd.read_csv(split_files[s]) for s in ['train', 'val', 'test'])
    print('Created and saved a new split.')

print(f'Train: {len(train_df):6d} images')
print(f'Val  : {len(val_df):6d} images')
print(f'Test : {len(test_df):6d} images')

## Step 1 — Download and clean the PlantDoc field photos
Web images come in many sizes and formats, and some are broken. We convert each one to a clean 224×224 JPEG, fix phone-photo rotation, and skip files that can't be opened.

In [ ]:
# PlantDoc folder name  ->  matching PlantVillage class
PLANTDOC_TO_PV = {
    'Apple Scab Leaf':            'Apple___Apple_scab',
    'Apple leaf':                 'Apple___healthy',
    'Apple rust leaf':            'Apple___Cedar_apple_rust',
    'Bell_pepper leaf':           'Pepper,_bell___healthy',
    'Bell_pepper leaf spot':      'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf':             'Blueberry___healthy',
    'Cherry leaf':                'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot':        'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight':           'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf':             'Corn_(maize)___Common_rust_',
    'Peach leaf':                 'Peach___healthy',
    'Potato leaf early blight':   'Potato___Early_blight',
    'Potato leaf late blight':    'Potato___Late_blight',
    'Raspberry leaf':             'Raspberry___healthy',
    'Soyabean leaf':              'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf':            'Strawberry___healthy',
    'Tomato Early blight leaf':   'Tomato___Early_blight',
    'Tomato Septoria leaf spot':  'Tomato___Septoria_leaf_spot',
    'Tomato leaf':                'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight':    'Tomato___Late_blight',
    'Tomato leaf mosaic virus':   'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus':   'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf':           'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf':                 'Grape___healthy',
    'grape leaf black rot':       'Grape___Black_rot',
}

In [ ]:
PD_DIR = os.path.join(DATA_ROOT, 'PlantDoc-Dataset')
if not os.path.isdir(PD_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pratikkayal/PlantDoc-Dataset', PD_DIR], check=True)

In [ ]:
from PIL import Image, ImageOps

CLEAN_DIR = os.path.join(DATA_ROOT, 'plantdoc_clean')

def build_plantdoc(split):
    rows, skipped = [], 0
    for folder in sorted(os.listdir(os.path.join(PD_DIR, split))):
        if folder not in PLANTDOC_TO_PV:
            continue
        label = CLASS_NAMES.index(PLANTDOC_TO_PV[folder])
        out_dir = os.path.join(CLEAN_DIR, split, folder)
        os.makedirs(out_dir, exist_ok=True)
        for i, fn in enumerate(sorted(os.listdir(os.path.join(PD_DIR, split, folder)))):
            out = os.path.join(out_dir, f'{i:04d}.jpg')
            if not os.path.exists(out):
                try:
                    with Image.open(os.path.join(PD_DIR, split, folder, fn)) as im:
                        im = ImageOps.exif_transpose(im).convert('RGB')
                        im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR).save(out, quality=95)
                except Exception:
                    skipped += 1
                    continue
            rows.append((out, folder, label))
    print(f'PlantDoc {split}: {len(rows)} images ({skipped} broken files skipped)')
    return pd.DataFrame(rows, columns=['path', 'plantdoc_class', 'label_id'])

pd_train_all = build_plantdoc('train')
pd_test = build_plantdoc('test')        # kept completely separate: final field score

In [ ]:
# Hold back 15% of PlantDoc-train as validation (to decide when to stop fine-tuning)
from sklearn.model_selection import train_test_split
counts = pd_train_all['label_id'].value_counts()
can_stratify = counts.min() >= 2
pd_train, pd_val = train_test_split(pd_train_all, test_size=0.15, random_state=SEED,
                                    stratify=pd_train_all['label_id'] if can_stratify else None)

# Lab photos: a small balanced sample so the model keeps its PlantVillage knowledge
lab_train = train_df.sample(frac=1, random_state=SEED).groupby('label_id').head(LAB_PER_CLASS)
lab_val = val_df.sample(frac=1, random_state=SEED).groupby('label_id').head(10)
lab_path = lambda frame: [os.path.join(DATASET_DIR, p) for p in frame['filepath']]

train_paths  = lab_path(lab_train) + list(pd_train['path']) * FIELD_REPEAT
train_labels = list(lab_train['label_id']) + list(pd_train['label_id']) * FIELD_REPEAT
val_paths    = lab_path(lab_val) + list(pd_val['path'])
val_labels   = list(lab_val['label_id']) + list(pd_val['label_id'])

mix = pd.DataFrame({
    'Lab photos (PlantVillage)':   [len(lab_train), len(lab_val), len(test_df)],
    'Field photos (PlantDoc)':     [len(pd_train), len(pd_val), len(pd_test)],
}, index=['Fine-tuning train', 'Fine-tuning validation', 'Final test'])
mix

## Step 2 — Data pipeline

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print('GPU available:', tf.config.list_physical_devices('GPU'))
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

def make_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((list(paths), np.array(labels, dtype=np.int64)))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

ft_train_ds = make_ds(train_paths, train_labels, shuffle=True)
ft_val_ds   = make_ds(val_paths, val_labels)
pv_test_ds  = make_ds(lab_path(test_df), test_df['label_id'])
pd_test_ds  = make_ds(pd_test['path'], pd_test['label_id'])

## Step 3 — Fine-tune MobileNetV2 on lab + field photos
- Start from the Notebook 2 model (it already knows the 38 diseases).
- Unfreeze the **top 60 layers** (more than before, so it can adapt to real backgrounds and lighting).
- Use a **small learning rate (3e-5)** so it adjusts gently instead of forgetting.
- The model's built-in augmentation (flip, rotate, zoom, contrast) stays active.

In [ ]:
orig_path  = os.path.join(MODELS_DIR, 'mobilenetv2.keras')
field_path = os.path.join(MODELS_DIR, 'mobilenetv2_field.keras')
field_hist_path = os.path.join(RESULTS_DIR, 'mobilenetv2_field_history.json')
assert os.path.exists(orig_path), 'mobilenetv2.keras not found - run Notebook 2 first.'
tf.keras.utils.set_random_seed(SEED)

if os.path.exists(field_path) and os.path.exists(field_hist_path) and not RETRAIN:
    print('Loading the already fine-tuned model from Drive...')
    field_model = keras.models.load_model(field_path)
    with open(field_hist_path) as f:
        d = json.load(f)
    field_history, field_seconds = d['history'], d['train_seconds']
else:
    field_model = keras.models.load_model(orig_path)
    base = next(l for l in field_model.layers if 'mobilenetv2' in l.name.lower())
    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    field_model.compile(optimizer=keras.optimizers.Adam(3e-5),
                        loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(field_path, monitor='val_accuracy', save_best_only=True),
    ]
    t0 = time.time()
    h = field_model.fit(ft_train_ds, validation_data=ft_val_ds, epochs=EPOCHS_FIELD, callbacks=callbacks)
    field_seconds = time.time() - t0
    field_history = {k: [float(v) for v in vals] for k, vals in h.history.items()}
    field_model.save(field_path)
    with open(field_hist_path, 'w') as f:
        json.dump({'history': field_history, 'train_seconds': field_seconds}, f)

print(f'Fine-tuning time: {field_seconds/60:.1f} min')

In [ ]:
epochs = np.arange(1, len(field_history['accuracy']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, m, label in [(axes[0], 'accuracy', 'Accuracy'), (axes[1], 'loss', 'Loss')]:
    ax.plot(epochs, field_history[m], 'o-', label=f'Training {label.lower()}')
    ax.plot(epochs, field_history['val_' + m], 's-', label=f'Validation {label.lower()}')
    ax.set_xlabel('Epoch'); ax.set_title(f'Field fine-tuning: {label}'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'field_finetune_curves.png'), dpi=150); plt.show()

## Step 4 — Before vs after
Both models are tested on the **same** photos: the PlantVillage test set (lab) and the PlantDoc test folder (real photos never used in training).

In [ ]:
orig_model = keras.models.load_model(orig_path)
probs = {}
for name, m in [('MobileNetV2 (before)', orig_model), ('MobileNetV2 + field fine-tuning (after)', field_model)]:
    probs[name] = {'lab': m.predict(pv_test_ds, verbose=1), 'field': m.predict(pd_test_ds, verbose=1)}

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

y_lab, y_field = test_df['label_id'].values, pd_test['label_id'].values
field_crop = np.array([crop_of(CLASS_NAMES[i]) for i in y_field])
is_tomato = field_crop == 'Tomato'

rows = []
for name, p in probs.items():
    lab_pred, field_pred = p['lab'].argmax(1), p['field'].argmax(1)
    pred_crop = np.array([crop_of(CLASS_NAMES[i]) for i in field_pred])
    rows.append({
        'Model': name,
        'Lab accuracy (%)': accuracy_score(y_lab, lab_pred) * 100,
        'Lab macro F1 (%)': f1_score(y_lab, lab_pred, average='macro') * 100,
        'Field accuracy (%)': accuracy_score(y_field, field_pred) * 100,
        'Field top-3 (%)': top_k_accuracy_score(y_field, p['field'], k=3, labels=np.arange(NUM_CLASSES)) * 100,
        'Field macro F1 (%)': f1_score(y_field, field_pred, average='macro', labels=np.unique(y_field), zero_division=0) * 100,
        'Field correct crop (%)': (pred_crop == field_crop).mean() * 100,
        'Field accuracy - tomato only (%)': accuracy_score(y_field[is_tomato], field_pred[is_tomato]) * 100 if is_tomato.any() else np.nan,
    })
ft_results = pd.DataFrame(rows).set_index('Model')
ft_results.to_csv(os.path.join(RESULTS_DIR, 'field_finetune_results.csv'))
print(f'Field test photos: {len(pd_test)}  (tomato: {int(is_tomato.sum())})')
ft_results.round(2)

In [ ]:
cols = ['Lab accuracy (%)', 'Field accuracy (%)', 'Field top-3 (%)']
ax = ft_results[cols].T.plot(kind='bar', figsize=(10, 5), rot=0, color=['#9e9e9e', '#2e7d32'], width=0.7)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f', padding=2)
plt.ylabel('%'); plt.ylim(0, 108); plt.title('Effect of fine-tuning on real field photos')
plt.legend(loc='upper right'); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'field_finetune_before_after.png'), dpi=150); plt.show()

In [ ]:
# Example real photos: prediction after fine-tuning
p = probs['MobileNetV2 + field fine-tuning (after)']['field']; pred = p.argmax(1)
rng = np.random.default_rng(SEED)
idx = rng.choice(len(pd_test), min(12, len(pd_test)), replace=False)
plt.figure(figsize=(16, 11))
for k, i in enumerate(idx):
    ok = pred[i] == y_field[i]
    plt.subplot(3, 4, k + 1); plt.imshow(Image.open(pd_test['path'].iloc[i])); plt.axis('off')
    plt.title(f'Pred: {SHORT[pred[i]]} ({p[i].max()*100:.0f}%)\nTrue: {SHORT[y_field[i]]}',
              color='green' if ok else 'red', fontsize=8.5)
plt.suptitle('After field fine-tuning: real photos (PlantDoc test)'); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'field_finetune_examples.png'), dpi=120); plt.show()

## ✅ What this shows
- Adding a few thousand **real field photos** to training lets the model learn that backgrounds, hands and lighting are *not* part of the disease, so real-world accuracy rises.
- Mixing in lab photos keeps the **lab accuracy** high (check the table: it should change only slightly).
- Real-world accuracy is still lower than lab accuracy, because PlantDoc is small (~2,300 training photos) and very varied. More field data is the key to further gains.

The web app in Notebook 3 now uses `mobilenetv2_field.keras` automatically.